In [1]:
import numpy as np
import json
from pathlib import Path
from datetime import datetime
import rasterio

# ============================================================
# CONFIGURATION
# ============================================================
ALIGNED_DATA_DIR = "/p/scratch/training2600/TeamGudnason/data/aligned_data"
OUTPUT_DIR       = "/p/scratch/training2600/TeamGudnason/training_data/final_block"

T33TYN_TILES = [
    "S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829",
    "S2A_MSIL2A_20180523T095031_N0500_R079_T33TYN_20230903T233859",
    "S2B_MSIL2A_20180816T095029_N0500_R079_T33TYN_20230812T013012",
    "S2A_MSIL2A_20181020T095031_N0500_R079_T33TYN_20230728T230408",
]

BLOCK_SIZE = 330   # ~1km x 1km blocks við 10m resolution
TRAIN_FRAC = 0.50
VAL_FRAC   = 0.25
TEST_FRAC  = 0.25

N_TRAIN = 100000
N_VAL   = 30000
N_TEST  = 30000

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("TIME SERIES MERGE + BLOCK CROSS-VALIDATION SPLIT")
print("=" * 70)

# ============================================================
# STEP 1: Extract patches with (row, col) coordinates
# ============================================================
def extract_patches_with_coords(s2_path, corine_path, patch_size=3,
                                 low_pct=2, high_pct=98):
    with rasterio.open(s2_path) as s2_src, rasterio.open(corine_path) as corine_src:
        height = s2_src.height
        width  = s2_src.width
        stride = patch_size
        s2_data     = s2_src.read()
        corine_data = corine_src.read(1)

        lo = np.percentile(s2_data, low_pct)
        hi = np.percentile(s2_data, high_pct)
        s2_norm = np.clip((s2_data.astype(np.float32) - lo) / (hi - lo + 1e-6), 0, 1)
        norm_params = {'method': 'percentile', 'low_pct': low_pct, 'high_pct': high_pct,
                       'low_val': float(lo), 'high_val': float(hi)}

        coord_to_patch = {}
        coord_to_label = {}

        for y in range(0, height - patch_size + 1, stride):
            for x in range(0, width - patch_size + 1, stride):
                center_y = y + patch_size // 2
                center_x = x + patch_size // 2

                label = corine_data[center_y, center_x]
                if label < 1 or label > 44:
                    continue

                orig_patch = s2_data[:, y:y+patch_size, x:x+patch_size]
                if np.any(orig_patch == 0):
                    continue

                patch = s2_norm[:, y:y+patch_size, x:x+patch_size]
                patch = np.transpose(patch, (1, 2, 0))

                coord_to_patch[(center_y, center_x)] = patch
                coord_to_label[(center_y, center_x)] = label

    return coord_to_patch, coord_to_label, height, width, norm_params

# ============================================================
# STEP 2: Load all tiles
# ============================================================
print("\n[1/4] Building coordinate maps...")
data_path = Path(ALIGNED_DATA_DIR)
tile_dicts, label_dicts, norm_params_all = [], [], []
img_height, img_width = None, None

for tile_name in T33TYN_TILES:
    s2_path     = data_path / f"{tile_name}_stacked.tif"
    corine_path = data_path / f"corine_aligned_{tile_name}.tif"
    print(f"  {tile_name[7:22]}...")
    c2p, c2l, h, w, np_ = extract_patches_with_coords(s2_path, corine_path)
    tile_dicts.append(c2p)
    label_dicts.append(c2l)
    norm_params_all.append(np_)
    if img_height is None:
        img_height = h
        img_width  = w

print(f"  Image size: {img_width} x {img_height}")

# ============================================================
# STEP 3: Merge into time series
# ============================================================
print("\n[2/4] Merging into time series patches (N, 3, 3, 16)...")
common_coords = set(tile_dicts[0].keys())
for td in tile_dicts[1:]:
    common_coords &= set(td.keys())
print(f"  Common locations: {len(common_coords):,}")

common_coords = sorted(common_coords)
ts_patches, ts_labels, ts_rows, ts_cols = [], [], [], []

for (row, col) in common_coords:
    band_stack = np.concatenate(
        [tile_dicts[t][(row, col)] for t in range(len(T33TYN_TILES))],
        axis=-1
    )
    ts_patches.append(band_stack)
    ts_labels.append(label_dicts[0][(row, col)])
    ts_rows.append(row)
    ts_cols.append(col)

ts_patches = np.array(ts_patches, dtype=np.float32)
ts_labels  = np.array(ts_labels,  dtype=np.uint8)
ts_rows    = np.array(ts_rows,    dtype=np.int32)
ts_cols    = np.array(ts_cols,    dtype=np.int32)

print(f"  Time series dataset: {ts_patches.shape}")

# ============================================================
# STEP 4: Block cross-validation split
# ============================================================
print(f"\n[3/4] Block cross-validation split (block size: {BLOCK_SIZE}px)...")

# Assign each patch to a block
block_row = ts_rows // BLOCK_SIZE
block_col = ts_cols // BLOCK_SIZE

n_blocks_y = img_height // BLOCK_SIZE
n_blocks_x = img_width  // BLOCK_SIZE
total_blocks = n_blocks_y * n_blocks_x
print(f"  Grid: {n_blocks_x} x {n_blocks_y} = {total_blocks} blocks")

# Assign blocks to splits deterministically
rng = np.random.default_rng(42)
block_ids = np.arange(total_blocks)
rng.shuffle(block_ids)

n_train_blocks = int(total_blocks * TRAIN_FRAC)
n_val_blocks   = int(total_blocks * VAL_FRAC)

train_blocks = set(block_ids[:n_train_blocks])
val_blocks   = set(block_ids[n_train_blocks:n_train_blocks + n_val_blocks])
test_blocks  = set(block_ids[n_train_blocks + n_val_blocks:])

print(f"  Train blocks: {len(train_blocks)}, Val: {len(val_blocks)}, Test: {len(test_blocks)}")

# Assign each patch to a split based on its block
patch_block_id = block_row * n_blocks_x + block_col

train_mask = np.array([int(b) in train_blocks for b in patch_block_id])
val_mask   = np.array([int(b) in val_blocks   for b in patch_block_id])
test_mask  = np.array([int(b) in test_blocks  for b in patch_block_id])

print(f"  Available — train: {train_mask.sum():,}, val: {val_mask.sum():,}, test: {test_mask.sum():,}")

# Sample from each split
def sample_split(patches, labels, mask, n, rng):
    idx = np.where(mask)[0]
    rng.shuffle(idx)
    idx = idx[:n]
    return patches[idx], labels[idx]

train_patches, train_labels = sample_split(ts_patches, ts_labels, train_mask, N_TRAIN, rng)
val_patches,   val_labels   = sample_split(ts_patches, ts_labels, val_mask,   N_VAL,   rng)
test_patches,  test_labels  = sample_split(ts_patches, ts_labels, test_mask,  N_TEST,  rng)

print(f"\n  Final splits:")
print(f"    Train: {train_patches.shape}")
print(f"    Val:   {val_patches.shape}")
print(f"    Test:  {test_patches.shape}")

# Check class distribution
def label_dist(labels):
    unique, counts = np.unique(labels, return_counts=True)
    return {int(k): int(v) for k, v in zip(unique, counts)}

print("\n  Class distribution comparison:")
tr_dist = label_dist(train_labels)
te_dist = label_dist(test_labels)
all_classes = sorted(set(tr_dist) | set(te_dist))
print(f"  {'CORINE':<10} {'Train%':>8} {'Test%':>8}")
for c in all_classes:
    tr_pct = 100 * tr_dist.get(c, 0) / len(train_labels)
    te_pct = 100 * te_dist.get(c, 0) / len(test_labels)
    if tr_pct > 0.5 or te_pct > 0.5:
        print(f"  {c:<10} {tr_pct:>8.1f}% {te_pct:>8.1f}%")

# ============================================================
# STEP 5: Save
# ============================================================
print("\n[4/4] Saving files...")
out = Path(OUTPUT_DIR)

np.savez_compressed(out / "patches_train.npz", patches=train_patches)
np.savez_compressed(out / "patches_val.npz",   patches=val_patches)
np.savez_compressed(out / "patches_test.npz",  patches=test_patches)
np.save(out / "labels_train.npy", train_labels)
np.save(out / "labels_val.npy",   val_labels)
np.save(out / "labels_test.npy",  test_labels)

metadata = {
    "extraction_timestamp": datetime.now().isoformat(),
    "source_tiles": T33TYN_TILES,
    "patch_size": 3,
    "n_bands_total": 16,
    "geographic_split": {
        "method": "block_cross_validation",
        "block_size_pixels": BLOCK_SIZE,
        "block_size_meters": BLOCK_SIZE * 10,
        "grid": f"{n_blocks_x} x {n_blocks_y}",
        "total_blocks": total_blocks,
        "train_blocks": len(train_blocks),
        "val_blocks": len(val_blocks),
        "test_blocks": len(test_blocks),
    },
    "splits": {
        "train": {"n_patches": len(train_labels), "label_distribution": label_dist(train_labels)},
        "val":   {"n_patches": len(val_labels),   "label_distribution": label_dist(val_labels)},
        "test":  {"n_patches": len(test_labels),  "label_distribution": label_dist(test_labels)},
    }
}

with open(out / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("  ✅ Allar skrár vistaðar")
print(f"\nOutput: {OUTPUT_DIR}")
for fname in ["patches_train.npz", "patches_val.npz", "patches_test.npz"]:
    fpath = out / fname
    if fpath.exists():
        print(f"  {fname}: {fpath.stat().st_size/(1024**2):.1f} MB")

TIME SERIES MERGE + BLOCK CROSS-VALIDATION SPLIT

[1/4] Building coordinate maps...
  L2A_20180408T09...
  L2A_20180523T09...
  L2A_20180816T09...
  L2A_20181020T09...
  Image size: 10980 x 10980

[2/4] Merging into time series patches (N, 3, 3, 16)...
  Common locations: 13,395,431
  Time series dataset: (13395431, 3, 3, 16)

[3/4] Block cross-validation split (block size: 330px)...
  Grid: 33 x 33 = 1089 blocks
  Train blocks: 544, Val: 272, Test: 273
  Available — train: 6,648,319, val: 3,317,549, test: 3,316,463

  Final splits:
    Train: (100000, 3, 3, 16)
    Val:   (30000, 3, 3, 16)
    Test:  (30000, 3, 3, 16)

  Class distribution comparison:
  CORINE       Train%    Test%
  2               6.4%      6.8%
  3               1.8%      1.6%
  11              1.2%      1.1%
  12             47.1%     47.7%
  15              1.1%      1.1%
  16              0.6%      0.4%
  18              5.6%      5.7%
  20              2.4%      2.8%
  21              1.8%      2.0%
  23       